# RTMPose-m fine-tune for 衣類キーポイント検出（tops + jacket）

**目的**: デタウリ/トルダケのアノテーション 154件で衣類キーポイント10点を学習し、Modal Serverless GPU にデプロイ可能な ONNX を出力する。

**前提**:
- Kaggle Notebook で **Accelerator = GPU P100** を選択（Settings → Accelerator）
- データセット `coco_tops_jacket` を Add data 経由でアタッチ済み（zip解凍されたものが `/kaggle/input/coco-tops-jacket/` に配置される想定）
- データセット名は Kaggle 側で kebab-case になる点に注意（`coco_tops_jacket` → `coco-tops-jacket`）

**所要時間目安**: セットアップ 5分 / 学習 30〜90分 / ONNX export 5分

**出力**: `/kaggle/working/rtmpose_garment_top.onnx`（Modal にアップロード）

## 1. GPU と環境の確認

In [ ]:
import sys, torch, platform
print('python      :', sys.version.split()[0])
print('platform    :', platform.platform())
print('torch       :', torch.__version__)
print('cuda avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device :', torch.cuda.get_device_name(0))
    print('vram (GB)   :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 2. MMPose と依存ライブラリのインストール

Kaggle の PyTorch 環境に合わせて mmcv のビルド版を選ぶ。`openmim` 経由なら CUDA バージョンを自動解決してくれる。

In [ ]:
# 既存環境を壊さないように --no-deps は付けず、openmim に解決させる
!pip install -q -U openmim
!mim install -q 'mmengine>=0.10.0'
!mim install -q 'mmcv>=2.0.1,<2.2.0'
!mim install -q 'mmdet>=3.1.0,<3.3.0'
!mim install -q 'mmpose>=1.1.0,<1.4.0'
# ONNX export 用
!pip install -q onnx onnxruntime-gpu onnxsim

In [ ]:
# インストール確認
import mmengine, mmcv, mmdet, mmpose, onnx
print('mmengine :', mmengine.__version__)
print('mmcv     :', mmcv.__version__)
print('mmdet    :', mmdet.__version__)
print('mmpose   :', mmpose.__version__)
print('onnx     :', onnx.__version__)

## 3. データセットを所定の場所に配置

MMPose の `CocoDataset` は `data_root/<ann_file>` と `data_root/<data_prefix>` のパス規約。`/kaggle/working/dataset/` に統一する。

In [ ]:
import os, shutil, glob
from pathlib import Path

# Kaggle データセット名の自動検出（kebab-case 変換ゆれに対応）
candidates = glob.glob('/kaggle/input/*/coco_tops_jacket') + glob.glob('/kaggle/input/coco-tops-jacket*/coco_tops_jacket') + glob.glob('/kaggle/input/coco_tops_jacket*')
if not candidates:
    raise FileNotFoundError('coco_tops_jacket dataset not found under /kaggle/input/. Did you attach the dataset?')
SRC = candidates[0]
DST = Path('/kaggle/working/dataset')
DST.mkdir(parents=True, exist_ok=True)

for sub in ('images', 'annotations'):
    src_dir = Path(SRC) / sub
    dst_dir = DST / sub
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)

print('src :', SRC)
print('dst :', DST)
print('train images :', len(list((DST/'images').glob('*'))))
import json
for split in ('train', 'val'):
    d = json.load(open(DST/'annotations'/f'{split}.json'))
    print(f'{split}: images={len(d["images"])} annotations={len(d["annotations"])}')

## 4. 学習 config を書き出す

RTMPose-m の Body8 pretrained config をベースに、衣類10点用にヘッドを差し替える。

In [ ]:
CONFIG_PY = r'''
_base_ = [
    'mmpose::_base_/default_runtime.py',
]

# ====== カスタム ======
dataset_type = 'CocoDataset'
data_root = '/kaggle/working/dataset/'
input_size = (256, 192)  # (W, H) - RTMPose 既定の半分。小データで過学習抑制
num_keypoints = 10

KEYPOINT_INFO = {
    0:  dict(name='collar_center',  id=0, color=[255,128,0], type='upper', swap=''),
    1:  dict(name='left_shoulder',  id=1, color=[51,153,255], type='upper', swap='right_shoulder'),
    2:  dict(name='right_shoulder', id=2, color=[51,153,255], type='upper', swap='left_shoulder'),
    3:  dict(name='left_armpit',    id=3, color=[0,255,0],    type='upper', swap='right_armpit'),
    4:  dict(name='right_armpit',   id=4, color=[0,255,0],    type='upper', swap='left_armpit'),
    5:  dict(name='left_cuff',      id=5, color=[255,128,255],type='upper', swap='right_cuff'),
    6:  dict(name='right_cuff',     id=6, color=[255,128,255],type='upper', swap='left_cuff'),
    7:  dict(name='hem_left',       id=7, color=[255,255,0],  type='lower', swap='hem_right'),
    8:  dict(name='hem_right',      id=8, color=[255,255,0],  type='lower', swap='hem_left'),
    9:  dict(name='hem_center',     id=9, color=[200,200,200],type='lower', swap=''),
}
SKELETON = [
    (1,2),(3,4),(5,6),(7,8),
    (0,1),(0,2),(1,3),(2,4),(3,5),(4,6),(3,7),(4,8),(7,9),(8,9),
]
dataset_info = dict(
    dataset_name='garment_top',
    paper_info=dict(author='Detauri/Toludake', title='Garment Top Keypoints'),
    keypoint_info=KEYPOINT_INFO,
    skeleton_info={i: dict(link=(KEYPOINT_INFO[a]['name'], KEYPOINT_INFO[b]['name']), id=i, color=[180,180,180]) for i,(a,b) in enumerate(SKELETON)},
    joint_weights=[1.0]*10,
    sigmas=[0.035]*10,
)

# ====== 学習スケジュール ======
max_epochs = 120
stage2_num_epochs = 20
base_lr = 4e-4
train_batch_size = 16
val_batch_size = 8

train_cfg = dict(max_epochs=max_epochs, val_interval=5)
auto_scale_lr = dict(base_batch_size=256)

optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr=base_lr, weight_decay=0.05),
    paramwise_cfg=dict(norm_decay_mult=0, bias_decay_mult=0, bypass_duplicate=True),
)
param_scheduler = [
    dict(type='LinearLR', begin=0, end=500, start_factor=0.001, by_epoch=False),
    dict(
        type='CosineAnnealingLR', eta_min=base_lr*0.05,
        begin=max_epochs//2, end=max_epochs, T_max=max_epochs//2,
        by_epoch=True, convert_to_iter_based=True,
    ),
]

# ====== モデル ======
codec = dict(type='SimCCLabel', input_size=input_size, sigma=(4.9, 5.66), simcc_split_ratio=2.0,
             normalize=False, use_dark=False)

model = dict(
    type='TopdownPoseEstimator',
    data_preprocessor=dict(
        type='PoseDataPreprocessor',
        mean=[123.675,116.28,103.53], std=[58.395,57.12,57.375], bgr_to_rgb=True),
    backbone=dict(
        _scope_='mmdet', type='CSPNeXt', arch='P5', expand_ratio=0.5, deepen_factor=0.67, widen_factor=0.75,
        out_indices=(4,), channel_attention=True, norm_cfg=dict(type='SyncBN'), act_cfg=dict(type='SiLU'),
        init_cfg=dict(type='Pretrained', prefix='backbone.',
            checkpoint='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.pth'),
    ),
    head=dict(
        type='RTMCCHead', in_channels=768, out_channels=num_keypoints,
        input_size=codec['input_size'], in_featuremap_size=tuple([s // 32 for s in codec['input_size']]),
        simcc_split_ratio=codec['simcc_split_ratio'],
        final_layer_kernel_size=7,
        gau_cfg=dict(hidden_dims=256, s=128, expansion_factor=2, dropout_rate=0., drop_path=0., act_fn='SiLU', use_rel_bias=False, pos_enc=False),
        loss=dict(type='KLDiscretLoss', use_target_weight=True, beta=10., label_softmax=True),
        decoder=codec),
    test_cfg=dict(flip_test=True),
)

# ====== データパイプライン ======
backend_args = dict(backend='local')
train_pipeline = [
    dict(type='LoadImage', backend_args=backend_args),
    dict(type='GetBBoxCenterScale'),
    dict(type='RandomFlip', direction='horizontal'),
    dict(type='RandomHalfBody'),
    dict(type='RandomBBoxTransform', scale_factor=[0.6,1.4], rotate_factor=30),
    dict(type='TopdownAffine', input_size=codec['input_size']),
    dict(type='mmdet.YOLOXHSVRandomAug'),
    dict(type='GenerateTarget', encoder=codec),
    dict(type='PackPoseInputs'),
]
val_pipeline = [
    dict(type='LoadImage', backend_args=backend_args),
    dict(type='GetBBoxCenterScale'),
    dict(type='TopdownAffine', input_size=codec['input_size']),
    dict(type='PackPoseInputs'),
]

train_dataloader = dict(
    batch_size=train_batch_size, num_workers=2, persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(type=dataset_type, data_root=data_root, data_mode='topdown',
        ann_file='annotations/train.json', data_prefix=dict(img='images/'),
        metainfo=dataset_info, pipeline=train_pipeline))
val_dataloader = dict(
    batch_size=val_batch_size, num_workers=2, persistent_workers=True, drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False, round_up=False),
    dataset=dict(type=dataset_type, data_root=data_root, data_mode='topdown',
        ann_file='annotations/val.json', data_prefix=dict(img='images/'),
        metainfo=dataset_info, pipeline=val_pipeline, test_mode=True))
test_dataloader = val_dataloader

val_evaluator = [
    dict(type='PCKAccuracy'),
    dict(type='AUC'),
    dict(type='EPE'),
    dict(type='CocoMetric', ann_file=data_root + 'annotations/val.json', use_area=False),
]
test_evaluator = val_evaluator

default_hooks = dict(
    checkpoint=dict(save_best='PCK', rule='greater', max_keep_ckpts=2),
    logger=dict(type='LoggerHook', interval=10),
)
custom_hooks = [
    dict(type='EMAHook', ema_type='ExpMomentumEMA', momentum=0.0002, update_buffers=True, priority=49),
    dict(type='mmdet.PipelineSwitchHook', switch_epoch=max_epochs-stage2_num_epochs,
         switch_pipeline=[
             dict(type='LoadImage', backend_args=backend_args),
             dict(type='GetBBoxCenterScale'),
             dict(type='RandomFlip', direction='horizontal'),
             dict(type='RandomBBoxTransform', scale_factor=[0.75,1.25], rotate_factor=10),
             dict(type='TopdownAffine', input_size=codec['input_size']),
             dict(type='mmdet.YOLOXHSVRandomAug'),
             dict(type='GenerateTarget', encoder=codec),
             dict(type='PackPoseInputs'),
         ]),
]

work_dir = '/kaggle/working/work_dir'
'''
config_path = Path('/kaggle/working/rtmpose_garment_top.py')
config_path.write_text(CONFIG_PY)
print('wrote', config_path, '({} bytes)'.format(config_path.stat().st_size))

## 5. 学習を実行

P100 で `max_epochs=120`、154件・batch=16 なら 1 epoch ≈ 10〜20秒 → トータル 30〜60分の見込み。Kaggle Notebook の実行制限（9時間）に余裕でおさまる。

In [ ]:
import os
os.environ['TORCH_HOME'] = '/kaggle/working/torch_cache'
# mmpose の学習スクリプトを直接呼ぶ
!python -m mmpose.utils.collect_env || true
!python -c "from mmpose.apis import init_model; print('mmpose ok')"

In [ ]:
# Trainer 実行
!python -m mmengine.tools.train /kaggle/working/rtmpose_garment_top.py 2>&1 | tail -200 || \
  python -c "from mmengine.runner import Runner; from mmengine.config import Config; cfg = Config.fromfile('/kaggle/working/rtmpose_garment_top.py'); Runner.from_cfg(cfg).train()"

In [ ]:
# 上の `!python -m mmengine.tools.train` が module not found の場合は↓を使う
from mmengine.runner import Runner
from mmengine.config import Config
cfg = Config.fromfile('/kaggle/working/rtmpose_garment_top.py')
runner = Runner.from_cfg(cfg)
runner.train()

## 6. ベスト checkpoint の確認

In [ ]:
import glob
ckpts = sorted(glob.glob('/kaggle/working/work_dir/best_PCK_*.pth'))
print('best ckpts:', ckpts)
BEST = ckpts[-1] if ckpts else None
if not BEST:
    # フォールバック：最終 epoch を使う
    epoch_ckpts = sorted(glob.glob('/kaggle/working/work_dir/epoch_*.pth'))
    BEST = epoch_ckpts[-1] if epoch_ckpts else None
print('using:', BEST)

## 7. ONNX エクスポート

mmdeploy 経由が公式だが Kaggle で重い。直接 `torch.onnx.export` で書く（推論時は SimCC head の argmax + decoder で keypoint 復元）。

In [ ]:
import torch
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmpose.apis import init_model

assert BEST, 'No checkpoint to export'
cfg = Config.fromfile('/kaggle/working/rtmpose_garment_top.py')
model = init_model(cfg, BEST, device='cuda:0')
model.eval()

# SimCC head は (B, K, W*split_ratio), (B, K, H*split_ratio) の2 tensor を返す
input_size = cfg.codec['input_size']  # (W, H) = (256, 192)
dummy = torch.randn(1, 3, input_size[1], input_size[0], device='cuda:0')

onnx_path = '/kaggle/working/rtmpose_garment_top.onnx'
torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=['input'],
    output_names=['simcc_x', 'simcc_y'],
    opset_version=17,
    dynamic_axes={'input': {0: 'batch'}, 'simcc_x': {0: 'batch'}, 'simcc_y': {0: 'batch'}},
)
print('exported:', onnx_path)
import os
print('size MB :', round(os.path.getsize(onnx_path)/1024/1024, 2))

In [ ]:
# ONNX 整合性 + simplify
import onnx
from onnxsim import simplify
model_onnx = onnx.load(onnx_path)
onnx.checker.check_model(model_onnx)
model_simp, ok = simplify(model_onnx)
if ok:
    onnx.save(model_simp, onnx_path)
    print('simplified ok')
else:
    print('simplify failed, using original')
print('final size MB :', round(__import__('os').path.getsize(onnx_path)/1024/1024, 2))

## 8. ONNX 推論のサニティチェック

val.json の最初の画像で推論し、ground truth との差分（pixel距離）を確認する。Modal にデプロイする前に「学習が回った形跡」を見ておく。

In [ ]:
import json, cv2, numpy as np, onnxruntime as ort, math
from pathlib import Path

val = json.load(open('/kaggle/working/dataset/annotations/val.json'))
img_meta = val['images'][0]
ann = next(a for a in val['annotations'] if a['image_id']==img_meta['id'])
img_path = Path('/kaggle/working/dataset/images') / img_meta['file_name']
bgr = cv2.imread(str(img_path))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
H, W = rgb.shape[:2]

# bbox をクロップして 256x192 にリサイズ
bx, by, bw, bh = ann['bbox']
cx, cy = bx + bw/2, by + bh/2
side = max(bw, bh) * 1.25
x0, y0 = int(cx - side/2), int(cy - side/2)
x1, y1 = int(cx + side/2), int(cy + side/2)
x0c, y0c = max(0,x0), max(0,y0)
x1c, y1c = min(W,x1), min(H,y1)
crop = np.zeros((y1-y0, x1-x0, 3), dtype=np.uint8)
crop[(y0c-y0):(y0c-y0)+(y1c-y0c), (x0c-x0):(x0c-x0)+(x1c-x0c)] = rgb[y0c:y1c, x0c:x1c]
tgt_w, tgt_h = 256, 192
resized = cv2.resize(crop, (tgt_w, tgt_h))

mean = np.array([123.675,116.28,103.53], dtype=np.float32)
std  = np.array([58.395,57.12,57.375], dtype=np.float32)
inp = (resized.astype(np.float32) - mean) / std
inp = inp.transpose(2,0,1)[None]  # NCHW

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
simcc_x, simcc_y = sess.run(None, {'input': inp.astype(np.float32)})
# SimCC: argmax で復元（split_ratio=2 → 入力長×2 の bins）
split_ratio = 2.0
px = simcc_x.argmax(-1)[0] / split_ratio  # (K,)
py = simcc_y.argmax(-1)[0] / split_ratio
# crop 座標→元画像座標へ
kp_pred = np.stack([px * (x1-x0)/tgt_w + x0, py * (y1-y0)/tgt_h + y0], axis=1)

# GT との差分
gt = np.array(ann['keypoints']).reshape(-1,3)[:,:2]
diffs = np.linalg.norm(kp_pred - gt, axis=1)
print('per-keypoint pixel error :')
for name, d in zip(['collar','L_sh','R_sh','L_ax','R_ax','L_cuff','R_cuff','hem_L','hem_R','hem_C'], diffs):
    print(f'  {name:10s} {d:6.1f} px')
print(f'mean error               : {diffs.mean():.1f} px')

## 9. ダウンロード

Kaggle Notebook 右側パネル → Output タブ → `rtmpose_garment_top.onnx` をクリックしてダウンロード（または `Save Version` 後に `kaggle kernels output` で取得）。

Modal に渡すファイル: `rtmpose_garment_top.onnx`（約 15〜20MB の見込み）

**次のステップ（katsu→Claude）**: ONNX を `/Users/katsu/saisun-repo/workers/annotate/out/` に置き、Claude に「Modal にデプロイして」と指示。